# Connecting to Microsoft Fabric SQL Analytics Endpoint

This notebook shows how to connect to a Fabric Lakehouse SQL Analytics Endpoint from Python.

## Prerequisites
1. ODBC Driver 18 for SQL Server installed on macOS
2. Python packages: `pyodbc`, `pandas`, `sqlalchemy`
3. Access to Fabric workspace with Dp900_lakehouse

## Step 1: Install ODBC Driver on macOS

Run this in your terminal (not in Jupyter):

```bash
# Install Homebrew if you don't have it
/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

# Install Microsoft ODBC Driver
brew tap microsoft/mssql-release https://github.com/Microsoft/homebrew-mssql-release
brew update
brew install msodbcsql18 mssql-tools18
```

## Step 2: Install Python Packages

In [ ]:
# Install required packages
!pip install pyodbc pandas sqlalchemy

## Step 3: Get Your Connection Details from Fabric

1. Go to [Microsoft Fabric](https://app.fabric.microsoft.com)
2. Navigate to your workspace
3. Click on **Dp900_lakehouse**
4. Switch to **SQL analytics endpoint** view (not the Lakehouse view)
5. Copy the **SQL connection string** from the top

It will look like:
```
your-workspace-name.pbidedicated.windows.net
```

## Method 1: Using pyodbc with Azure AD Interactive Authentication

In [ ]:
import pyodbc
import pandas as pd

# ============ REPLACE WITH YOUR VALUES ============
SERVER = 'your-workspace-name.pbidedicated.windows.net'  # Get from Fabric portal
DATABASE = 'Dp900_lakehouse'
# ==================================================

# Connection string for Azure AD Interactive (browser popup)
conn_string = f"""
    Driver={{ODBC Driver 18 for SQL Server}};
    Server={SERVER};
    Database={DATABASE};
    Authentication=ActiveDirectoryInteractive;
    Encrypt=yes;
    TrustServerCertificate=no;
"""

try:
    # Connect to the database
    print("Attempting to connect...")
    conn = pyodbc.connect(conn_string)
    print("✓ Connection successful!")
    
    # Test query - list all tables
    query = """
    SELECT 
        TABLE_SCHEMA,
        TABLE_NAME,
        TABLE_TYPE
    FROM INFORMATION_SCHEMA.TABLES
    ORDER BY TABLE_SCHEMA, TABLE_NAME
    """
    
    df = pd.read_sql(query, conn)
    print(f"\nFound {len(df)} tables:")
    print(df)
    
    conn.close()
    
except pyodbc.Error as e:
    print(f"❌ Connection failed: {e}")
    print("\nTroubleshooting:")
    print("1. Make sure ODBC Driver 18 is installed: brew list msodbcsql18")
    print("2. Check your SERVER name from Fabric portal")
    print("3. Ensure you have access to the Fabric workspace")
    print("4. Try authenticating with your browser when prompted")

## Method 2: Using SQLAlchemy

In [ ]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import pandas as pd

# ============ REPLACE WITH YOUR VALUES ============
SERVER = 'your-workspace-name.pbidedicated.windows.net'
DATABASE = 'Dp900_lakehouse'
# ==================================================

# Build connection string
params = quote_plus(
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"Authentication=ActiveDirectoryInteractive;"
    f"Encrypt=yes;"
    f"TrustServerCertificate=no;"
)

connection_url = f"mssql+pyodbc:///?odbc_connect={params}"

try:
    # Create engine
    engine = create_engine(connection_url)
    
    # Test connection
    with engine.connect() as conn:
        print("✓ Connection successful!")
        
        # Query data
        query = "SELECT TOP 10 * FROM INFORMATION_SCHEMA.TABLES"
        df = pd.read_sql(query, conn)
        print(df)
        
except Exception as e:
    print(f"❌ Connection failed: {e}")

## Method 3: Connection Function for Reuse

In [ ]:
import pyodbc
import pandas as pd
from typing import Optional

class FabricSQLConnection:
    """Helper class for connecting to Fabric SQL Analytics Endpoint."""
    
    def __init__(self, server: str, database: str = 'Dp900_lakehouse'):
        self.server = server
        self.database = database
        self.conn = None
    
    def connect(self) -> pyodbc.Connection:
        """Establish connection to Fabric SQL Analytics Endpoint."""
        conn_string = f"""
            Driver={{ODBC Driver 18 for SQL Server}};
            Server={self.server};
            Database={self.database};
            Authentication=ActiveDirectoryInteractive;
            Encrypt=yes;
            TrustServerCertificate=no;
        """
        
        try:
            self.conn = pyodbc.connect(conn_string)
            print(f"✓ Connected to {self.database}")
            return self.conn
        except pyodbc.Error as e:
            raise ConnectionError(f"Failed to connect to Fabric SQL: {e}")
    
    def query(self, sql: str) -> pd.DataFrame:
        """Execute SQL query and return results as DataFrame."""
        if not self.conn:
            self.connect()
        return pd.read_sql(sql, self.conn)
    
    def list_tables(self) -> pd.DataFrame:
        """List all tables in the lakehouse."""
        query = """
        SELECT 
            TABLE_SCHEMA as [Schema],
            TABLE_NAME as [Table],
            TABLE_TYPE as [Type]
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA != 'INFORMATION_SCHEMA'
        ORDER BY TABLE_SCHEMA, TABLE_NAME
        """
        return self.query(query)
    
    def close(self):
        """Close the connection."""
        if self.conn:
            self.conn.close()
            print("✓ Connection closed")


# Usage example
# ============ REPLACE WITH YOUR VALUES ============
SERVER = 'your-workspace-name.pbidedicated.windows.net'
# ==================================================

# Create connection
fabric = FabricSQLConnection(SERVER)

try:
    # List tables
    tables = fabric.list_tables()
    print("\nAvailable tables:")
    print(tables)
    
    # Query specific table (replace with your table name)
    # df = fabric.query("SELECT TOP 100 * FROM your_table_name")
    # print(df.head())
    
finally:
    fabric.close()

## Common Connection Errors and Solutions

### Error: "TCP Provider, error: 35"
**Solution**: This usually means:
1. ODBC Driver not installed → Install using homebrew (see Step 1)
2. Wrong server name → Get correct name from Fabric portal
3. Network issues → Check firewall/VPN

### Error: "Named Pipes Provider"
**Solution**: Add `Encrypt=yes;` to connection string

### Error: "Login failed"
**Solution**: 
- Make sure you have access to the Fabric workspace
- Use `Authentication=ActiveDirectoryInteractive` for browser login
- Check your Azure AD credentials

### Error: "Driver not found"
**Solution**: 
```bash
# Verify driver installation
odbcinst -q -d -n "ODBC Driver 18 for SQL Server"

# If not found, install:
brew install msodbcsql18
```

### Error: "SSL Security error"
**Solution**: Add `TrustServerCertificate=yes;` (for dev only) or install proper certificates

## Verify ODBC Driver Installation

In [ ]:
import pyodbc

# List all available ODBC drivers
drivers = [driver for driver in pyodbc.drivers()]
print("Available ODBC drivers:")
for driver in drivers:
    print(f"  - {driver}")

# Check for SQL Server driver
sql_drivers = [d for d in drivers if 'SQL Server' in d]
if sql_drivers:
    print(f"\n✓ SQL Server drivers found: {sql_drivers}")
else:
    print("\n❌ No SQL Server driver found!")
    print("Install with: brew install msodbcsql18")